# Notebook 1 — Data Loading, Shuffling & Splitting

## Purpose
This notebook is the **first step** in the pipeline. It:
1. Loads all 4,505 raw SisFall trials from Google Drive
2. Randomly shuffles trials to break subject ordering
3. Splits trials 70 / 15 / 15 into train, val, and test — **test set is locked immediately**
4. Saves the split trial data to `split_output.npz`

**Why shuffle before splitting?**
SisFall files are stored per-subject (SA01, SA02, ...). Without shuffling, a sequential split
would put all of SA01's trials in train, SA02's in val, etc. — concentrating subjects in single
splits. Shuffling ensures windows from every subject are distributed across all three splits,
giving each split the same population characteristics.

**Why lock the test set here?**
All subsequent design decisions (thresholds, ADL selection rules) must be derived from
train+val only. Locking the test set in this first notebook prevents any accidental leakage.

**Output:** `split_output.npz` → loaded by Notebook 2 (threshold analysis) and all experiment notebooks.

---
## Requirements
```
scikit-learn>=1.2.0  numpy>=1.23.0
```


## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Install dependencies

In [2]:
!pip install scikit-learn -q
print('Done')

Done


## Cell 3 — Configuration
All paths, sensor constants, and split parameters are defined here.
Change `DATASET_DIR` if your SisFall folder is in a different location.

In [10]:
import os
import json
import random
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict

# ── Paths ──────────────────────────────────────────────────────────────
DATASET_DIR   = '/content/drive/MyDrive/M4/Data_Collected'
SPLIT_PATH    = '/content/drive/MyDrive/M4/01_split.npz'
ANALYSIS_JSON = '/content/drive/MyDrive/M4/02_analysis.json'

# ── Constants ──────────────────────────────────────────────────────────
TRIM_SAMPLES = 100   # rows trimmed from start/end of each file
WIN_SAMPLES  = 50    # 500 ms at 100 Hz
STRIDE       = 20    # 200 ms stride (60% overlap)

# ── Split parameters ───────────────────────────────────────────────────
RANDOM_SEED = 42
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Volunteer lists ────────────────────────────────────────────────────
# V1–V8  : volunteers who performed falls AND ADLs
# V10    : Isham — falls + ADLs
ALL_VOLUNTEERS = [
    'V1_Sama', 'V2_Shams', 'V3_Mary', 'V4_Yasamin',
    'V5_Betty', 'V6_Jannah', 'V7_Warda', 'V8_Ismat',
    'V10_Isham'
]
FALL_VOLUNTEERS = ALL_VOLUNTEERS   # all volunteers performed falls
ADL_VOLUNTEERS  = ALL_VOLUNTEERS   # all volunteers performed ADLs

# ── Activity codes ─────────────────────────────────────────────────────
# Derived from filenames: F01–F19 falls, D01–D19 ADLs
# (exact range auto-detected from files)
import glob
_all_csvs = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
fall_codes = sorted(set(
    os.path.basename(f)[:3] for f in _all_csvs
    if os.path.basename(f)[0].upper() == 'F'
))
adl_codes = sorted(set(
    os.path.basename(f)[:3] for f in _all_csvs
    if os.path.basename(f)[0].upper() == 'D'
))

FALL_CODES = fall_codes   # e.g. ['F01', 'F02', ..., 'F19']
ADL_CODES  = adl_codes    # e.g. ['D01', 'D02', ..., 'D19']

# ── EXCLUDE — files with no detectable impact or parse errors ──────────
EXCLUDE = {
    'V1_Sama/D11_01.csv', 'V1_Sama/F04_05.csv',
    'V1_Sama/D07_03.csv', 'V1_Sama/D17_01.csv',
    'V1_Sama/D02_01.csv', 'V1_Sama/F01_01.csv',
    'V10_Isham/F01_01.csv', 'V10_Isham/F01_02.csv', 'V10_Isham/F01_03.csv',
    'V10_Isham/F02_02.csv', 'V10_Isham/F03_02.csv', 'V10_Isham/F03_03.csv',
    'V10_Isham/F03_04.csv', 'V10_Isham/F04_01.csv', 'V10_Isham/F06_01.csv',
    'V10_Isham/F09_01.csv',
    'V2_Shams/F02_01.csv', 'V2_Shams/F03_02.csv',
    'V2_Shams/F07_01.csv', 'V2_Shams/F14_01.csv',
    'V3_Mary/F02_01.csv',  'V3_Mary/F02_02.csv',
    'V3_Mary/F03_01.csv',  'V3_Mary/F03_02.csv',
    'V4_Yasamin/F01_01.csv', 'V4_Yasamin/F03_01.csv',
    'V5_Betty/F03_01.csv',   'V5_Betty/F11_01.csv',
    'V6_Jannah/F05_01.csv',  'V6_Jannah/F06_01.csv',
    'V6_Jannah/F07_01.csv',  'V6_Jannah/F09_01.csv',
    'V6_Jannah/F12_01.csv',
    'V7_Warda/F01_01.csv',  'V7_Warda/F01_03.csv',
    'V7_Warda/F04_01.csv',  'V7_Warda/F05_01.csv',
    'V7_Warda/F06_01.csv',  'V7_Warda/F07_01.csv',
    'V7_Warda/F09_01.csv',  'V7_Warda/F12_01.csv',
}

print('Configuration set')
print(f'  Volunteers : {len(ALL_VOLUNTEERS)}')
print(f'  Fall codes : {len(FALL_CODES)} → {FALL_CODES}')
print(f'  ADL codes  : {len(ADL_CODES)}  → {ADL_CODES}')
print(f'  Excluded   : {len(EXCLUDE)} files')

Configuration set
  Volunteers : 9
  Fall codes : 16 → ['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'fal']
  ADL codes  : 19  → ['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19']
  Excluded   : 41 files


## Cell 4 — Helper functions

### `load_trial(filepath)`
Reads one SisFall `.txt` file. Each file contains one activity recording.
The file has 9 comma-separated columns per row, semicolon-terminated.
We use only two sensors:
- **MMA8451Q** (cols 7–9): accelerometer ±8g — matches Arduino LSM9DS1 default range
- **ITG3200** (cols 4–6): gyroscope ±2000 dps

### `downsample(signal)`
SisFall was recorded at 200 Hz. The Arduino runs at 100 Hz natively.
We take every other sample (factor-of-2 decimation) to match Arduino's rate.
No anti-aliasing filter is applied — experiments confirmed identical model
performance vs a Chebyshev filter approach.

### `smv(a)`
Signal Magnitude Vector = √(x² + y² + z²). A single scalar per sample
that captures overall motion intensity regardless of orientation.


In [11]:
import pandas as pd

def load_trial(filepath):
    """
    Read one collected CSV file and return physical-unit sensor data.
    Files are already cleaned with header: t_ms,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps
    Data is already in physical units (g and dps) — no scaling needed.
    First and last TRIM_SAMPLES rows are removed to avoid recording artifacts.

    Returns:
      accel [N, 3] in g    (ax_g, ay_g, az_g)
      gyro  [N, 3] in dps  (gx_dps, gy_dps, gz_dps)
      Returns (None, None) if file has fewer than WIN_SAMPLES rows.
    """
    TRIM_SAMPLES = 100
    cols = ['t_ms','ax_g','ay_g','az_g','gx_dps','gy_dps','gz_dps']

    header_line = None
    with open(filepath, 'rb') as f:
        for i, line in enumerate(f):
            if line.strip().startswith(b't_ms'):
                header_line = i; break
    if header_line is None:
        return None, None

    try:
        df = pd.read_csv(filepath, skiprows=header_line, on_bad_lines='skip')
        for c in cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        df = df.dropna(subset=cols)[['ax_g','ay_g','az_g','gx_dps','gy_dps','gz_dps']]
    except Exception as e:
        print(f'  Parse error {filepath}: {e}')
        return None, None

    df = df.iloc[TRIM_SAMPLES:-TRIM_SAMPLES]
    if len(df) < WIN_SAMPLES:
        return None, None

    accel = df[['ax_g','ay_g','az_g']].values.astype(np.float32)
    gyro  = df[['gx_dps','gy_dps','gz_dps']].values.astype(np.float32)
    return accel, gyro


def downsample(signal):
    """
    NOT needed for collected data.
    Arduino LSM9DS1 is configured natively at 100 Hz — data is already
    at 100 Hz, unlike SisFall which was recorded at 200 Hz.
    This function is kept as a no-op for API compatibility.
    """
    return signal.astype(np.float32)   # pass-through, no decimation


def smv(a):
    """Signal Magnitude Vector: sqrt(x² + y² + z²) per sample."""
    return np.sqrt(np.sum(a ** 2, axis=1))


print('Helper functions defined')

Helper functions defined


## Cell 5 — Phase 1: Load all raw trials

Loads every `.txt` file from all 38 subjects into memory as a list of trial dicts.
Each dict contains the subject ID, activity code (e.g. F01, D11), raw downsampled
accelerometer and gyroscope arrays, and pre-computed SMV arrays.

No windowing or labeling happens here — just raw signal loading.


In [12]:
print('Phase 1 — Loading all raw trials...')
print()

all_trials = []  # list of dicts, one per .csv file

for volunteer in ALL_VOLUNTEERS:
    vol_dir = os.path.join(DATASET_DIR, volunteer)
    if not os.path.isdir(vol_dir):
        print(f'  ⚠ Volunteer folder not found: {vol_dir}')
        continue

    for fname in sorted(os.listdir(vol_dir)):
        if not fname.endswith('.csv'):
            continue

        # Check exclude list
        rel_path = f'{volunteer}/{fname}'
        if rel_path in EXCLUDE:
            continue

        fpath         = os.path.join(vol_dir, fname)
        activity_code = fname.split('_')[0]   # e.g. 'F01', 'D11'
        is_fall       = activity_code in FALL_CODES

        accel_raw, gyro_raw = load_trial(fpath)
        if accel_raw is None:
            print(f'  ⚠ Skipped (too short or parse error): {rel_path}')
            continue

        # No downsampling needed — already at 100 Hz
        a_ds = downsample(accel_raw)
        g_ds = downsample(gyro_raw)

        all_trials.append({
            'volunteer':     volunteer,
            'filename':      fname,
            'activity_code': activity_code,
            'is_fall':       is_fall,
            'accel_ds':      a_ds,
            'gyro_ds':       g_ds,
            'a_smv':         smv(a_ds),
            'g_smv':         smv(g_ds),
        })

fall_trials = [t for t in all_trials if     t['is_fall']]
adl_trials  = [t for t in all_trials if not t['is_fall']]

print(f'Loaded {len(all_trials):,} trials total')
print(f'  Fall trials : {len(fall_trials):,}')
print(f'  ADL trials  : {len(adl_trials):,}')
print()

# Per-volunteer summary
print(f'{"Volunteer":<20} {"Falls":>6} {"ADLs":>6}')
print('-' * 35)
vol_counts = defaultdict(lambda: {'falls': 0, 'adls': 0})
for t in all_trials:
    if t['is_fall']:
        vol_counts[t['volunteer']]['falls'] += 1
    else:
        vol_counts[t['volunteer']]['adls'] += 1
for vol in ALL_VOLUNTEERS:
    f = vol_counts[vol]['falls']
    d = vol_counts[vol]['adls']
    print(f'{vol:<20} {f:>6} {d:>6}')

Phase 1 — Loading all raw trials...

  ⚠ Skipped (too short or parse error): V2_Shams/D17_01.csv
  ⚠ Skipped (too short or parse error): V7_Warda/D02_01.csv
  ⚠ Skipped (too short or parse error): V10_Isham/D11_01.csv
  ⚠ Skipped (too short or parse error): V10_Isham/F04_05.csv
Loaded 290 trials total
  Fall trials : 111
  ADL trials  : 179

Volunteer             Falls   ADLs
-----------------------------------
V1_Sama                  10     27
V2_Shams                 23     30
V3_Mary                   7     11
V4_Yasamin                9     11
V5_Betty                  9     11
V6_Jannah                22     30
V7_Warda                 18     15
V8_Ismat                  0     14
V10_Isham                13     30


## Cell 6 — Phase 2: Shuffle trials and split 70/15/15

**Step 1 — Shuffle:**
Fall trials and ADL trials are shuffled independently using `random.seed(42)`.
`random_state=42` makes the split reproducible but does NOT reduce randomness —
it is equivalent to setting a random seed before shuffling a deck of cards.

**Step 2 — Split:**
Trials (not windows) are split 70/15/15. This means all windows extracted later
from one trial will belong to the same split — no leakage within a trial.

**Step 3 — Lock test set:**
Test trials are never used again in this notebook or in Notebook 2.
They are saved to the NPZ and only loaded by the experiment notebooks
for final evaluation.


In [13]:
print('Phase 2 — Trial-wise split (shuffle then 70/15/15)...')
print()

random.seed(RANDOM_SEED)

# ── Shuffle trials ─────────────────────────────────────────────────────
fall_shuffled = fall_trials.copy()
adl_shuffled  = adl_trials.copy()
random.shuffle(fall_shuffled)
random.shuffle(adl_shuffled)

# ── Split fall trials 70 / 15 / 15 ────────────────────────────────────
fall_tv, fall_test = train_test_split(
    fall_shuffled, test_size=TEST_RATIO, random_state=RANDOM_SEED)
fall_train, fall_val = train_test_split(
    fall_tv, test_size=VAL_RATIO / (1 - TEST_RATIO), random_state=RANDOM_SEED)

# ── Split ADL trials 70 / 15 / 15 ─────────────────────────────────────
adl_tv, adl_test = train_test_split(
    adl_shuffled, test_size=TEST_RATIO, random_state=RANDOM_SEED)
adl_train, adl_val = train_test_split(
    adl_tv, test_size=VAL_RATIO / (1 - TEST_RATIO), random_state=RANDOM_SEED)

print(f'Fall  — Train: {len(fall_train):4d}  Val: {len(fall_val):3d}  Test: {len(fall_test):3d}')
print(f'ADL   — Train: {len(adl_train):4d}  Val: {len(adl_val):3d}  Test: {len(adl_test):3d}')
print()
print('TEST SET LOCKED — will not be used for any design decision')

Phase 2 — Trial-wise split (shuffle then 70/15/15)...

Fall  — Train:   77  Val:  17  Test:  17
ADL   — Train:  125  Val:  27  Test:  27

TEST SET LOCKED — will not be used for any design decision


## Cell 7 — Phase 3: Save split to Drive

Saves the split trial data as a compressed NPZ file.

**What is saved:**
- All six trial lists (fall_train, fall_val, fall_test, adl_train, adl_val, adl_test)
  as arrays of downsampled accelerometer + gyroscope signals
- The split keys (subject + activity_code) for reproducibility verification

**What is NOT saved:**
- Windows (windowing happens in each experiment notebook)
- Labels (labeling strategy varies per experiment)
- Thresholds (derived in Notebook 2)


In [14]:
print('Phase 3 — Saving split to Drive...')

def pack_trials(trial_list):
    """
    Convert a list of trial dicts into arrays for NPZ storage.
    Stores: accel_ds, gyro_ds, volunteer list, activity_code list.
    """
    accels     = [t['accel_ds']      for t in trial_list]
    gyros      = [t['gyro_ds']       for t in trial_list]
    volunteers = [t['volunteer']     for t in trial_list]
    codes      = [t['activity_code'] for t in trial_list]
    return accels, gyros, volunteers, codes

# Pack each split
fa,  fg,  fv,  fc  = pack_trials(fall_train)
va,  vg,  vv,  vc  = pack_trials(fall_val)
ta,  tg,  tv,  tc  = pack_trials(fall_test)
aa,  ag,  av,  ac  = pack_trials(adl_train)
ava, avg, avv, avc = pack_trials(adl_val)
ata, atg, atv, atc = pack_trials(adl_test)

split_path = SPLIT_PATH + '.npz' if not SPLIT_PATH.endswith('.npz') else SPLIT_PATH

np.savez_compressed(
    split_path,
    # Fall splits
    fall_train_accels     = np.array(fa,  dtype=object),
    fall_train_gyros      = np.array(fg,  dtype=object),
    fall_train_volunteers = np.array(fv),
    fall_train_codes      = np.array(fc),
    fall_val_accels       = np.array(va,  dtype=object),
    fall_val_gyros        = np.array(vg,  dtype=object),
    fall_val_volunteers   = np.array(vv),
    fall_val_codes        = np.array(vc),
    fall_test_accels      = np.array(ta,  dtype=object),
    fall_test_gyros       = np.array(tg,  dtype=object),
    fall_test_volunteers  = np.array(tv),
    fall_test_codes       = np.array(tc),
    # ADL splits
    adl_train_accels      = np.array(aa,  dtype=object),
    adl_train_gyros       = np.array(ag,  dtype=object),
    adl_train_volunteers  = np.array(av),
    adl_train_codes       = np.array(ac),
    adl_val_accels        = np.array(ava, dtype=object),
    adl_val_gyros         = np.array(avg, dtype=object),
    adl_val_volunteers    = np.array(avv),
    adl_val_codes         = np.array(avc),
    adl_test_accels       = np.array(ata, dtype=object),
    adl_test_gyros        = np.array(atg, dtype=object),
    adl_test_volunteers   = np.array(atv),
    adl_test_codes        = np.array(atc),
)

size_mb = os.path.getsize(split_path) / (1024 ** 2)
print(f'Saved: {split_path}  ({size_mb:.1f} MB)')
print()
print(f'Split summary (trial-wise):')
print(f'  Fall  — Train: {len(fall_train):3d}  Val: {len(fall_val):3d}  Test: {len(fall_test):3d}')
print(f'  ADL   — Train: {len(adl_train):3d}  Val: {len(adl_val):3d}  Test: {len(adl_test):3d}')
print()
print('Next step: run Notebook 2 (threshold analysis) using train+val splits.')

Phase 3 — Saving split to Drive...
Saved: /content/drive/MyDrive/M4/01_split.npz  (8.8 MB)

Split summary (trial-wise):
  Fall  — Train:  77  Val:  17  Test:  17
  ADL   — Train: 125  Val:  27  Test:  27

Next step: run Notebook 2 (threshold analysis) using train+val splits.
